In [1]:
import cytnx
import numpy as np

In [2]:
path = '../Networks'

def ut_print(ut, print_numpy=True):
    ut.print_diagram()
    if print_numpy:
        print(ut.get_block().numpy())

def get_M(T):
    W = np.array([[np.exp(1/T), np.exp(-1/T)],
                  [np.exp(-1/T), np.exp(1/T)]])
    W = cytnx.from_numpy(W)
    S, U, Vd = cytnx.linalg.Svd(W)
    M = U @ cytnx.linalg.Diag(S.Pow(0.5))
    M = cytnx.UniTensor(M, rowrank=1)
    M.set_name('M')
    Md = cytnx.linalg.Diag(S.Pow(0.5)) @ Vd
    Md = cytnx.UniTensor(Md, rowrank=1)
    Md.set_name('Md')
    return M, Md

def get_delta(h):
    delta = cytnx.zeros([2, 2, 2, 2])
    delta[0, 0, 0, 0] = 1
    delta[1, 1, 1, 1] = 1
    delta = cytnx.UniTensor(delta, rowrank=2)
    delta.set_name('delta')
    return delta

def get_T_baret(T, h=0):
    ut_M, ut_Md = get_M(T)
    ut_delta = get_delta(h)
    Ising_net = cytnx.Network(f'{path}/Ising_square.net')
    Ising_net.PutUniTensors(['delta', 'M0.d', 'M1.d', 'M2', 'M3'],
                            [ut_delta, ut_Md, ut_Md, ut_M, ut_M])
    T_bare = Ising_net.Launch()
    T_bare.set_name('T_bare')
    return T_bare

def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        TupTdn_net = cytnx.Network(f'{path}/merge_y_trace.net')
    else:
        TupTdn_net = cytnx.Network(f'{path}/merge_y.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn'], [Tup, Tdn])
    TupTdn = TupTdn_net.Launch()
    if combine:
        TupTdn.combineBonds(['1', '2'])
        TupTdn.combineBonds(['3', '4'])
    return TupTdn

def merge_y_truncate(Tup, Tdn, dcut):
    if ((Tup.shape()[1] * Tdn.shape()[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    TupTdn_pure_net = cytnx.Network(f'{path}/merge_y_pure.net')
    TupTdn_pure_net.PutUniTensors(['Tup', 'Tdn', 'Tupd', 'Tdnd'],
                                  [Tup, Tdn, Tup, Tdn])
    TupTdn_pure = TupTdn_pure_net.Launch()
    _, U, __ = cytnx.linalg.Svd_truncate(TupTdn_pure, dcut)
    TupTdn_net = cytnx.Network(f'{path}/merge_y_truncate.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn', 'UL', 'UR'],
                             [Tup, Tdn, U, U])
    TupTdn = TupTdn_net.Launch()
    return TupTdn

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        n = TL.shape()[0]
        I = np.eye(n)
        I = cytnx.from_numpy(I)
        I = cytnx.UniTensor(I, rowrank=1)
        TLTR_net = cytnx.Network(f'{path}/merge_x_trace.net')
        TLTR_net.PutUniTensors(['TL', 'TR', 'IL', 'IR'], [TL, TR, I, I])
        TLTR = TLTR_net.Launch()
    else:
        TLTR_net = cytnx.Network(f'{path}/merge_x.net')
        TLTR_net.PutUniTensors(['TL', 'TR'], [TL, TR])
        TLTR = TLTR_net.Launch()
    if combine and not trace:
        TLTR.combineBonds(['0', '1'])
        TLTR.combineBonds(['4', '5'])
    return TLTR

def merge_x_truncate(TL, TR, dcut):
    if ((TL.shape()[0] * TR.shape()[0]) < dcut):
        return merge_x(TL, TR, True)
    TLTR_pure_net = cytnx.Network(f'{path}/merge_x_pure.net')
    TLTR_pure_net.PutUniTensors(['TL', 'TR', 'TLd', 'TRd'],
                                [TL, TR, TL, TR])
    TLTR_pure = TLTR_pure_net.Launch()
    try:
        _, U, __ = cytnx.linalg.Svd_truncate(TLTR_pure, dcut)
    except:
        TLTR_pure.print_diagram()
        _, U, __ = cytnx.linalg.Svd(TLTR_pure, dcut)
    TLTR_net = cytnx.Network(f'{path}/merge_x_truncate.net')
    TLTR_net.PutUniTensors(['TL', 'TR', 'Uup', 'Udn'], [TL, TR, U, U])
    TLTR = TLTR_net.Launch()
    return TLTR

def trace(T):
    n = T.shape()[0]
    I = np.eye(n)
    I = cytnx.from_numpy(I)
    I = cytnx.UniTensor(I, rowrank=1)
    trace_net = cytnx.Network(f'{path}/trace.net')
    trace_net.PutUniTensors(['T', 'I'], [T, I])
    return trace_net.Launch()

In [3]:
M, Md = get_M(4)
print(M)
print(Md)

-------- start of print ---------
Tensor name: M
is_diag    : False
contiguous : True

Total elem: 4
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2)
[[-1.01559e+00 -5.02606e-01 ]
 [-1.01559e+00 5.02606e-01 ]]




-------- start of print ---------
Tensor name: Md
is_diag    : False
contiguous : True

Total elem: 4
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2)
[[-1.01559e+00 -1.01559e+00 ]
 [-5.02606e-01 5.02606e-01 ]]






In [4]:
print(get_T_baret(4))
print(get_T_baret(3))
print(get_T_baret(2))

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.12763e+00 0.00000e+00 ]
   [0.00000e+00 5.21095e-01 ]]
  [[1.11022e-16 5.21095e-01 ]
   [5.21095e-01 2.77556e-17 ]]]
 [[[1.11022e-16 5.21095e-01 ]
   [5.21095e-01 2.77556e-17 ]]
  [[5.21095e-01 5.55112e-17 ]
   [5.55112e-17 1.27626e-01 ]]]]




-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.23058e+00 3.33067e-16 ]
   [3.33067e-16 7.17158e-01 ]]
  [[1.11022e-16 7.17158e-01 ]
   [7.17158e-01 2.77556e-16 ]]]
 [[[1.11022e-16 7.17158e-01 ]
   [7.17158e-01 2.77556e-16 ]]
  [[7.17158e-01 1.11022e-16 ]
   [1.11022e-16 2.30576e-01 ]]]]




-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2

# No dcut

In [7]:
MaxL = 4
Temp = np.linspace(2.26,4,5000)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [8]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TL_Trace = merge_y(TL,TL,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y(TL,TL,True)
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 4min 9s, sys: 1min 2s, total: 5min 11s
Wall time: 49.3 s


<timed exec>:10: RuntimeWarning: divide by zero encountered in divide


In [7]:
Corr_len

array([[        inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,     

# dcut = 2

In [32]:
%%time
dcut = 4

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    print(TL)
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        print(TLx)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        print(TL)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
        print(TL)
    break
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])

-------- start of print ---------
Tensor name: T_bare
is_diag    : False
contiguous : True

Total elem: 16
type  : Double (Float64)
cytnx device: CPU
Shape : (2,2,2,2)
[[[[2.41780e+00 7.77156e-16 ]
   [6.66134e-16 1.00507e+00 ]]
  [[2.22045e-16 1.00507e+00 ]
   [1.00507e+00 1.11022e-16 ]]]
 [[[2.22045e-16 1.00507e+00 ]
   [1.00507e+00 1.11022e-16 ]]
  [[1.00507e+00 -1.11022e-16 ]
   [-1.11022e-16 4.17805e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 64
type  : Double (Float64)
cytnx device: CPU
Shape : (4,2,2,4)
[[[[6.03633e+00 1.42412e-16 -4.09153e-17 2.50390e-01 ]
   [8.94878e-15 2.60554e+00 8.72328e-01 -1.04636e-15 ]]
  [[7.25537e-15 2.60554e+00 -8.72328e-01 -8.13183e-16 ]
   [1.43744e+00 8.17440e-16 -6.49610e-16 -6.93442e-01 ]]]
 [[[-1.50671e-15 3.44024e+00 5.77316e-15 -1.06762e-15 ]
   [2.60554e+00 -5.12224e-15 -1.73371e-15 -3.89080e-01 ]]
  [[2.60554e+00 -5.81820e-15 1.08654e-15 -3.89080e-01 ]
   [5.43225e-16 1.430

   [1.67468e+00 -9.63784e-16 1.47037e-15 4.60139e-17 ]
   [-2.03576e-15 1.13613e+00 4.06368e-01 7.10301e-16 ]]
  [[6.25046e-15 1.11860e+00 1.54117e-01 -1.22049e-15 ]
   [1.67468e+00 -1.42405e-15 -1.59829e-15 -3.45104e-17 ]
   [-5.37808e-01 -1.63441e-15 -8.55729e-17 -9.18761e-02 ]
   [-9.76725e-16 -4.06368e-01 -1.93537e-01 7.79051e-16 ]]
  [[8.37111e-01 -8.95468e-16 -1.21991e-15 -3.75989e-01 ]
   [-3.38797e-15 1.13613e+00 -4.06368e-01 8.18649e-16 ]
   [-4.95712e-16 -4.06368e-01 1.93537e-01 7.79051e-16 ]
   [-3.66254e-01 1.58687e-15 2.72943e-16 2.29905e-01 ]]]]




-------- start of print ---------
Tensor name: 
is_diag    : False
contiguous : False

Total elem: 256
type  : Double (Float64)
cytnx device: CPU
Shape : (4,4,4,4)
[[[[1.06537e+03 9.84310e-13 5.12599e-14 7.20990e+01 ]
   [5.98212e-12 5.16556e+02 -2.27748e+02 -1.36185e-12 ]
   [6.13917e-14 8.34056e+01 -1.09056e+02 -6.85608e-13 ]
   [3.57236e+01 -8.42946e-13 7.57086e-13 3.74647e+01 ]]
  [[5.41962e-12 5.16556e+02 2.27748e+02 -1.1

Total elem: 256
type  : Double (Float64)
cytnx device: CPU
Shape : (4,4,4,4)
[[[[2.30286e+01 2.30944e-13 1.34699e-15 1.15020e+00 ]
   [5.84099e-14 1.20036e+01 -3.78276e+00 -6.21885e-14 ]
   [2.18682e-15 3.77175e+00 -3.55314e+00 -4.23553e-14 ]
   [-1.44112e+00 3.56469e-14 -4.17846e-14 -1.30448e+00 ]]
  [[4.42603e-14 1.20036e+01 3.78276e+00 -5.95786e-14 ]
   [8.91322e+00 -2.04810e-13 8.30902e-15 -3.49833e+00 ]
   [4.33484e-15 -3.89212e-14 2.19996e-14 -2.47163e+00 ]
   [1.52484e-14 7.08679e-01 -1.61197e+00 -6.31205e-15 ]]
  [[2.18682e-15 3.77175e+00 3.55314e+00 -3.82958e-14 ]
   [5.55401e-15 -3.56484e-14 -1.47388e-14 -2.47163e+00 ]
   [1.24704e+00 -1.30765e-14 1.10454e-15 -1.10162e+00 ]
   [6.70735e-16 -8.86050e-01 -1.75108e-02 -4.33402e-15 ]]
  [[-1.44112e+00 2.91637e-14 4.21228e-14 -1.30448e+00 ]
   [9.00649e-15 7.08679e-01 1.61197e+00 -2.68092e-15 ]
   [6.70735e-16 -8.86050e-01 1.75108e-02 -2.80386e-15 ]
   [5.72851e-01 -1.09923e-14 1.43023e-15 -4.81168e-01 ]]]
 [[[2.10545e-13 8.90666e

   [-7.95324e+04 -5.17384e-10 1.18943e-09 -3.75274e+04 ]
   [-2.61738e-09 -7.51424e+04 -6.09062e+04 2.27930e-09 ]]
  [[2.22033e+05 -7.20002e-09 1.24187e-09 -1.74045e+04 ]
   [-1.73335e-08 1.80216e+05 1.09139e-10 2.79942e-09 ]
   [3.06173e-09 4.36557e-11 6.09062e+04 2.69185e-09 ]
   [-1.47376e+04 4.28660e-09 3.03358e-09 5.05903e+04 ]]
  [[-7.95324e+04 -5.17384e-10 -1.33125e-09 -3.75274e+04 ]
   [2.94370e-09 4.91127e-11 -6.09062e+04 2.85000e-09 ]
   [-1.77295e-09 -4.14510e+04 -1.09139e-11 1.04217e-09 ]
   [-2.02293e+04 9.14902e-11 1.24785e-10 2.21981e+04 ]]
  [[-2.42719e-09 -7.51424e+04 6.09062e+04 2.10494e-09 ]
   [-1.47376e+04 4.34294e-09 -2.77197e-09 5.05903e+04 ]
   [-2.02293e+04 9.14902e-11 1.75917e-11 2.21981e+04 ]
   [-6.78578e-10 -1.14317e+04 -2.72848e-11 2.44905e-10 ]]]
 [[[1.36991e-12 1.70985e-10 2.50274e+04 -8.18420e-12 ]
   [7.77272e+04 -1.22625e-09 -2.83215e-09 1.84480e+04 ]
   [-8.27933e+04 -1.79967e-09 -1.06857e-10 -3.92258e+01 ]
   [-3.71116e-09 -6.09062e+04 -2.99624e+04 

<timed exec>:19: RuntimeWarning: divide by zero encountered in divide


In [18]:
Corr_len_2

array([[       inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,     

# dcut = 4

In [6]:
%%time
dcut = 4

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_4 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 95.3 ms, sys: 2.18 ms, total: 97.5 ms
Wall time: 94.2 ms


<timed exec>:15: RuntimeWarning: divide by zero encountered in divide


# dcut = 10

In [7]:
%%time
dcut = 10

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_10 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 4.18 s, sys: 5.62 s, total: 9.8 s
Wall time: 1.59 s


<timed exec>:15: RuntimeWarning: divide by zero encountered in divide


# dcut = 20

In [8]:
%%time
dcut = 20

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_20 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 2min 42s, sys: 1min 3s, total: 3min 45s
Wall time: 30.5 s


<timed exec>:15: RuntimeWarning: divide by zero encountered in divide


In [9]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "Corr_len": Corr_len,
    "Corr_len_2": Corr_len_2,
    "Corr_len_4": Corr_len_4,
    "Corr_len_10": Corr_len_10,
    "Corr_len_20": Corr_len_20
}

with open("Corr_len.pkl", "wb") as file:
    pickle.dump(data, file)

## $Max = 128\times 128$

In [10]:
MaxL = 12
dcut = 20
Temp = np.linspace(2.26,2.36,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [11]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 1h 23min 57s, sys: 29min 43s, total: 1h 53min 40s
Wall time: 13min 15s


<timed exec>:12: RuntimeWarning: divide by zero encountered in divide


In [12]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "dcut": dcut,
    "Corr_len": Corr_len
}

with open("Corr_len_20.pkl", "wb") as file:
    pickle.dump(data, file)

In [13]:
MaxL = 12
dcut = 16
Temp = np.linspace(2.26,2.36,250)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [14]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    del TL, TLx
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
CPU times: user 44min 23s, sys: 18min 28s, total: 1h 2min 51s
Wall time: 8min 56s


<timed exec>:14: RuntimeWarning: divide by zero encountered in divide


In [15]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "dcut": dcut,
    "Corr_len": Corr_len
}

with open("Corr_len_16.pkl", "wb") as file:
    pickle.dump(data, file)

# Secant_Method

In [16]:
def f(temp, MaxL, dcut = 20):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    TL_Trace = merge_y(TLx,TLx,True,True)
    w1,_ = cytnx.linalg.Eigh(TL_Trace.get_block())
    E1 = -np.log(w1.numpy()[-2:])
    Corr_len1 = 1/(E1[-2]-E1[-1])
    TLx = merge_x_truncate(TL,TL,dcut)
    TL = merge_y_truncate(TLx,TLx,dcut)
    TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    TL_Trace = merge_y(TLx,TLx,True,True)
    w2,_ = cytnx.linalg.Eigh(TL_Trace.get_block())
    E2 = -np.log(w2.numpy()[-2:])
    Corr_len2 = 1/(E2[-2]-E2[-1])
    return Corr_len1-Corr_len2

def Secant_Method(func,x0,x1,Maxiter,MaxL,dcut):
    x = [x0,x1]
    func_out = [func(x0,MaxL,dcut)]
    for i in range(Maxiter):
        if (x[-1] - x[-2]) == 0:
            break
        func_out.append(func(x[-1],MaxL,dcut))
        if (func_out[-1] - func_out[-2]) == 0:
            break
        if np.abs(func_out[-1]) <= 1e-13:
            break
        new_x = x[-1] - func_out[-1] * (x[-1] - x[-2]) / (func_out[-1] - func_out[-2])
        x.append(new_x)
        #print(i,x[-1],func_out[-1])
    return x[-1]

In [17]:
MaxL = 22
dcuts = [4, 6, 8, 12, 16, 20]
Tc = 2/np.log(1+np.sqrt(2))
T_stars = []

In [18]:
%%time
for dcut in dcuts:
    T_star = [2.36]
    print("dcut =",dcut)
    for i in range(2,MaxL):
        #print(i , 2**(i-1))
        T_star.append(Secant_Method(f,2.26,T_star[-1],10,i,dcut))
        print(i , 2**(i-1),T_star[-1])
    T_stars.append(T_star)
T_stars = np.array(T_stars)

dcut = 4
2 2 2.3850858946981073
3 4 2.2868821123754204
4 8 2.2614197710826027
5 16 2.2507999018944895
6 32 2.2458448192152622
7 64 2.2436082949919807
8 128 2.2426586516563574
9 256 2.4036459643368797
10 512 2.403014540424002
11 1024 2.4039021592344487
12 2048 2.404353887021841
13 4096 2.3657050964090205
14 8192 2.302786370359246
15 16384 2.3027863681861556
16 32768 2.308896361915139
17 65536 2.278132152163173
18 131072 2.265087303489002
19 262144 2.2759528725945053
20 524288 2.27590928259243
21 1048576 2.2686222153818156
dcut = 6
2 2 2.3471529615798046
3 4 2.2859185962603696
4 8 2.2769242297418946
5 16 2.2749270386678737
6 32 2.273458581048248
7 64 2.272410544612897
8 128 2.271790779121379
9 256 2.2714758877429384
10 512 2.2713326959844413
11 1024 2.2712725885924403
12 2048 2.2712487673997095
13 4096 2.27123970586174
14 8192 2.2712363562236577
15 16384 2.2712351416606786
16 32768 2.2712347065153105
17 65536 2.2712345515699632
18 131072 2.271234496470452
19 262144 2.271234476822966
20 5

In [19]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "dcuts": dcuts,
    "T_stars": T_stars,
}

with open("Secant_Method.pkl", "wb") as file:
    pickle.dump(data, file)